In [2]:
from sodapy import Socrata
import pandas as pd

In [5]:
# Cliente anónimo — sin API key para exploración inicial
client = Socrata("www.datos.gov.co", None)

DATASET_PRECIPITACION = "s54a-sgyg"

# Muestra pequeña para ver el formato real
muestra = client.get(DATASET_PRECIPITACION, limit=5)
df = pd.DataFrame.from_records(muestra)
print(f"Columnas: {list(df.columns)}")
print(f"Filas: {len(df)}")
print(df.to_string())

Columnas: ['codigoestacion', 'codigosensor', 'fechaobservacion', 'valorobservado', 'nombreestacion', 'departamento', 'municipio', 'zonahidrografica', 'latitud', 'longitud', 'descripcionsensor', 'unidadmedida']
Filas: 5
  codigoestacion codigosensor         fechaobservacion valorobservado        nombreestacion  departamento    municipio zonahidrografica      latitud      longitud descripcionsensor unidadmedida
0     0024037190         0240  2019-08-18T19:40:00.000              0     SAN RAFAEL  - AUT        BOYACÁ     TIBASOSA         SOGAMOSO  5.807611111  -73.01380556     Precipitacion           mm
1     2120000108         0240  2018-12-14T08:56:00.000              0          IDIGER - AUT   BOGOTA D.C.  BOGOTA, D.C   ALTO MAGDALENA        4.675       -74.114     Precipitacion           mm
2     0021201200         0240  2016-02-11T06:50:00.000              0  ESC LA UNION - FOPAE   BOGOTA D.C.  BOGOTA, D.C   ALTO MAGDALENA        4.343       -74.184     Precipitacion           mm
3    

In [6]:
# Estrategia: filtrar por departamento en vez de contar todo
# Los datasets de IDEAM son enormes — las queries sin filtro hacen timeout

# Estaciones en Antioquia (donde están Porce II, Porce III, Ituango)
muestra_antioquia = client.get(
    DATASET_PRECIPITACION,
    where="departamento='ANTIOQUIA'",
    limit=10,
)

df_ant = pd.DataFrame.from_records(muestra_antioquia)
print(f"Filas: {len(df_ant)}")
print(df_ant[['nombreestacion','fechaobservacion',
              'valorobservado','municipio']].to_string())

Filas: 10
                 nombreestacion         fechaobservacion valorobservado              municipio
0         CIUDAD BOLIVAR  - AUT  2018-03-28T06:30:00.000              0         CIUDAD BOLÍVAR
1         HACIENDA COTOVE - AUT  2018-12-22T16:10:00.000              0  SANTA FE DE ANTIOQUIA
2            CAÑASGORDAS  - AUT  2017-10-11T20:30:00.000              0            CAÑASGORDAS
3       ARGELIA ANTIOQUIA - AUT  2015-09-01T09:20:00.000              0                ARGELIA
4                  ARAGON - AUT  2019-04-16T00:50:00.000              0     SANTA ROSA DE OSOS
5             EL ROSARIO  - AUT  2018-01-22T03:25:00.000              0                VENECIA
6           SANTA BARBARA - AUT  2017-11-17T19:20:00.000              0          SANTA BÁRBARA
7            PISTA INDIRA - AUT  2011-12-31T07:40:00.000              0                  TURBO
8  APTO OLAYA HERRERA - TX GPRS  2018-05-09T09:30:00.000              0               MEDELLÍN
9                   MACEO - AUT  2011-12

In [7]:
# Filtrar por zona hidrográfica Y fecha reciente
# Alto Magdalena es la zona más crítica para los embalses colombianos

from datetime import date, timedelta

fecha_inicio = "2024-01-01"
fecha_fin = "2024-12-31"

muestra_magdalena = client.get(
    DATASET_PRECIPITACION,
    where=f"zonahidrografica='ALTO MAGDALENA' \
            AND fechaobservacion >= '{fecha_inicio}T00:00:00.000' \
            AND fechaobservacion <= '{fecha_fin}T23:59:59.000'",
    limit=20,
)

df_mag = pd.DataFrame.from_records(muestra_magdalena)
print(f"Filas: {len(df_mag)}")
if not df_mag.empty:
    print(f"\nZonas hidrográficas disponibles en muestra:")
    print(df_mag['zonahidrografica'].unique())
    print(f"\nEstaciones únicas:")
    print(df_mag['nombreestacion'].unique())
    print(f"\nRango de fechas en muestra:")
    print(f"  Desde: {df_mag['fechaobservacion'].min()}")
    print(f"  Hasta: {df_mag['fechaobservacion'].max()}")
    print(f"\nMuestra de datos:")
    print(df_mag[['nombreestacion','fechaobservacion',
                  'valorobservado','municipio']].head(10).to_string())

Filas: 20

Zonas hidrográficas disponibles en muestra:
['ALTO MAGDALENA']

Estaciones únicas:
['LIBANO EL' 'ZULUAGA' 'PUENTE SANTANDER' 'SANTA MARIA' 'ROSALES LOS'
 'PURIFICACION' 'ALTAMIRA EL GRIFO' 'POTOSI HACIENDA' 'ARGENTINA LA'
 'FLORIDA LA' 'VALLE DE SAN JUAN' 'PIEDRAS']

Rango de fechas en muestra:
  Desde: 2024-01-01T00:00:00.000
  Hasta: 2024-01-01T00:10:00.000

Muestra de datos:
      nombreestacion         fechaobservacion valorobservado     municipio
0          LIBANO EL  2024-01-01T00:00:00.000              0         SUAZA
1            ZULUAGA  2024-01-01T00:00:00.000              0        GARZÓN
2   PUENTE SANTANDER  2024-01-01T00:00:00.000              0         NEIVA
3        SANTA MARIA  2024-01-01T00:00:00.000              0   SANTA MARÍA
4        ROSALES LOS  2024-01-01T00:00:00.000              0   CAMPOALEGRE
5       PURIFICACION  2024-01-01T00:00:00.000              0  PURIFICACIÓN
6  ALTAMIRA EL GRIFO  2024-01-01T00:00:00.000              0      ALTAMIRA
7    POT

In [8]:
# Identificar todas las zonas hidrográficas disponibles
# Para mapear cuáles cubren nuestros embalses críticos

zonas = client.get(
    DATASET_PRECIPITACION,
    select="zonahidrografica",
    group="zonahidrografica",
    limit=100,
)

df_zonas = pd.DataFrame.from_records(zonas)
print("Zonas hidrográficas disponibles:")
for zona in sorted(df_zonas['zonahidrografica'].tolist()):
    print(f"  {zona}")

ReadTimeout: HTTPSConnectionPool(host='www.datos.gov.co', port=443): Read timed out. (read timeout=10)

In [9]:
import requests

# Prueba Open-Meteo: datos históricos para la ubicación del embalse El Quimbo
# Coordenadas: 2.0797° N, 75.7625° W (Huila, Colombia)

url = "https://archive.open-meteo.com/v1/archive"

params = {
    "latitude": 2.0797,
    "longitude": -75.7625,
    "start_date": "2024-01-01",
    "end_date": "2024-01-31",
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min", 
        "temperature_2m_mean",
        "precipitation_sum",
    ],
    "timezone": "America/Bogota",
}

response = requests.get(url, params=params, timeout=30)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    df_clima = pd.DataFrame(data["daily"])
    print(f"Filas: {len(df_clima)}")
    print(f"Columnas: {list(df_clima.columns)}")
    print(df_clima.head(10).to_string())
else:
    print(f"Error: {response.text}")

ConnectionError: HTTPSConnectionPool(host='archive.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=2.0797&longitude=-75.7625&start_date=2024-01-01&end_date=2024-01-31&daily=temperature_2m_max&daily=temperature_2m_min&daily=temperature_2m_mean&daily=precipitation_sum&timezone=America%2FBogota (Caused by NameResolutionError("HTTPSConnection(host='archive.open-meteo.com', port=443): Failed to resolve 'archive.open-meteo.com' ([Errno -2] Name or service not known)"))

# EDA Clima — Exploración de fuentes de datos

## Decisión de fuente de datos

### IDEAM vía Socrata (datos.gov.co) — DESCARTADO
- Timeouts frecuentes en queries de agregación
- Datos desactualizados (muestras de 2011-2018)
- Granularidad de 10 minutos requiere agregación adicional
- Ver decisión completa en docs/decisiones_tecnicas.md #10

### Open-Meteo — FUENTE SELECCIONADA
- Confirmado funcional vía HTTP desde WSL
- Histórico desde 1940, pronóstico 16 días
- Ver cliente de producción en src/data/openmeteo_client.py

---
Las celdas siguientes documentan la exploración de IDEAM
que llevó a esta decisión.